# M5 lab: size and run a microgrid that survives an outage

*Energy Systems, Step by Step* · Modeler tier · lab for lesson M5 · about 60 minutes, plus 90 minutes of practice at the end.

You will rebuild the lesson's engine in Python and solve it with the open-source HiGHS solver through `scipy.optimize.milp`:

1. Prairie Campus, its year on the grid and the winter storm of 20 to 22 January 2025.
2. The sizing model: solar, a battery and whole generator sets at the least annual cost, with the storm's must-stay-on loads served.
3. Branch and bound on the sets, by hand.
4. The feeder's export limit, the time-of-use tariff, and the plan without the storm.
5. The check: the plan hour by hour with whole sets that run at 30% or more; then ten drawn winter outages.
6. The storm on forecasts with a rolling horizon: what forecast error costs.
7. Your own plan (the Modeler version of capstone C1).

**How to run it.** Put this notebook and `e1-prairie-campus.json` (download it from the lesson page) in one folder. In Google Colab: *File → Upload notebook*, then upload the data file with the folder icon on the left, then *Runtime → Run all*. Locally: Python 3.9 or later with `numpy` and `scipy` 1.9 or later. Everything runs in under a minute.

**The data.** Prairie Campus is fictional (lesson F7's): four flat loads, 1,200 kW in all. Its sun is a teaching profile per kW of panels: SPP's observed 2025 hourly solar (EIA-930), scaled to EIA's monthly capacity factors; not measured campus output. The outages are hypothetical: the grid assumed down in those hours. Equipment costs, fuel use, the export credit and the owners' costs of capital are illustrative, in 2025 dollars. Results are illustrative, not recommendations.

**What the asserts check.**

- Sizes, annual costs, the tree, the check and the drawn outages: against the lesson's reference numbers (the data file's `reference.m5`, computed by the page's engine).
- The rolling horizon: its physics in every real hour (supply meets the loads served, the battery and the tank stay in bounds, a set that is on runs at 30% to 100%). Its totals are printed beside the page's, not asserted: its re-plans can tie, and a different tie can change what happens in the real hours.

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import scipy
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import coo_matrix

major, minor = (int(x) for x in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this lab needs SciPy 1.9 or later (pip install -U scipy)"
assert Path("e1-prairie-campus.json").exists(), "e1-prairie-campus.json not found: put it next to this notebook (in Colab, upload it with the folder icon)."
D = json.load(open("e1-prairie-campus.json", encoding="utf-8"))
REF = D["reference"]["m5"]
MD = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
LOADS = D["campus"]["loads"]
TOTAL = sum(l["kw"] for l in LOADS)
T = D["tech"]
print("NumPy", np.__version__, "· SciPy", scipy.__version__)
print("Loads:", ", ".join(f'{l["label"]} {l["kw"]} kW' for l in LOADS), f"· {TOTAL:,} kW in all")

## 1. The campus, its year and the storm

- **The year on the grid** is 12 average days, one per month (the mean of each hour over the month), each weighted by its month's days.
- **The storm** is 72 hours from 00:00 on 20 January 2025: day 19 of the stored winter, `sun["winter"]["kw"][19*24 : 22*24]`.
- **The tariff**: time-of-use (16:00 to 21:00 at three times the other hours) or flat, both set so a flat load pays EIA's 2025 U.S. commercial average.
- **Costs a year**: capital times the capital recovery factor at the owner's real rate, plus upkeep.

In [ ]:
SUN_MONTHS = D["sun"]["months"]
WINTER = D["sun"]["winter"]["kw"]
STORM = WINTER[19 * 24: 22 * 24]
print("storm hours with no sun:", sum(1 for v in STORM if v == 0), "of", len(STORM))

def rate(plan, h):
    tt = D["grid"]["tariffs"]
    if plan["tar"] == "flat":
        return tt["flat"]["rate"]
    return tt["tou"]["on"] if tt["tou"]["peak"][0] <= h < tt["tou"]["peak"][1] else tt["tou"]["off"]

def crf(r, n):
    a = (1 + r) ** n
    return r * a / (a - 1)

def per_unit(plan):
    r = next(o["rate"] for o in D["owners"] if o["key"] == plan["own"])
    return {"pv": T["pv"]["capex"] * crf(r, T["pv"]["life"]) + T["pv"]["fom"],
            "bat": T["bat"]["capex"] * (crf(r, T["bat"]["life"]) + T["bat"]["fomShare"]),
            "gen": T["gen"]["unitKW"] * (T["gen"]["capex"] * crf(r, T["gen"]["life"]) + T["gen"]["fom"]),
            "tank": T["tank"]["capexPerGal"] * crf(r, T["tank"]["life"])}

BASE = {"pv": 600, "bat": 2000, "gen": 2, "fuel": 675, "prio": {"hospital": "must", "water": "must", "dorms": "keep", "labs": "shed"},
        "exp": 2000, "imp": 2000, "tar": "flat", "own": "community"}
pu = per_unit(BASE)
print({k: round(v, 2) for k, v in pu.items()}, "dollars a year per kW, kWh, set and gallon")
base_bill = sum(MD[m] * rate(BASE, h) / 100 * TOTAL for m in range(12) for h in range(24))
print(f"the year's bill with nothing built (flat tariff): ${base_bill:,.0f}")

## 2. A small model builder

The page builds its models as lists of named variables and rules and hands them to HiGHS. This cell does the same for `scipy.optimize.milp`: each variable has bounds, a cost and an integrality flag; each rule a set of `{name: coefficient}` terms and bounds.

In [ ]:
class Model:
    def __init__(self):
        self.names, self.idx, self.lb, self.ub, self.cost, self.integ = [], {}, [], [], [], []
        self.rows, self.rlo, self.rhi, self.offset = [], [], [], 0.0

    def var(self, name, lb=0.0, ub=math.inf, cost=0.0, integer=False):
        self.idx[name] = len(self.names); self.names.append(name)
        self.lb.append(lb); self.ub.append(ub); self.cost.append(cost); self.integ.append(1 if integer else 0)

    def con(self, terms, lo=-math.inf, hi=math.inf):
        self.rows.append(terms); self.rlo.append(lo); self.rhi.append(hi)

    def solve(self, relax=False, gap=1e-6, time_limit=60):
        r, c, v = [], [], []
        for i, terms in enumerate(self.rows):
            for k, a in terms.items():
                r.append(i); c.append(self.idx[k]); v.append(a)
        A = coo_matrix((v, (r, c)), shape=(len(self.rows), len(self.names))).tocsr()
        res = milp(np.array(self.cost), integrality=None if relax else np.array(self.integ),
                   bounds=Bounds(np.array(self.lb), np.array(self.ub)), constraints=LinearConstraint(A, np.array(self.rlo), np.array(self.rhi)),
                   options={"mip_rel_gap": gap, "time_limit": time_limit, "disp": False})
        out = {"ok": res.status == 0, "status": res.message, "objective": None, "x": None}
        if res.x is not None:
            out["objective"] = float(res.fun) + self.offset
            out["x"] = dict(zip(self.names, res.x))
        return out

def checked(r, what, provisional=False):
    # use a result only once its status is known: no solution stops the cell with what to do; a feasible plan that HiGHS
    # did not prove optimal (a time limit) stops it too, except in the rolling horizon (provisional=True), where it is
    # used and said to be provisional, as the page labels such a row
    if r["x"] is None:
        raise RuntimeError(f"{what}: HiGHS returned no solution ({r['status']}). Run the cell again, or raise time_limit in solve().")
    if not r["ok"]:
        if not provisional:
            raise RuntimeError(f"{what}: a feasible plan, not proven optimal ({r['status']}). Raise time_limit in solve() and run the cell again.")
        print(f"note: {what} is feasible but not proven optimal ({r['status']}): the totals that use it are provisional")
    return r

## 3. The sizing model

Every hour lasts $\Delta t = 1$ h, so a kW held for an hour is a kWh, and $B/4$ is the battery's power in kW.

| Symbol | What it is | Unit | Decided or given |
|---|---|---|---|
| $S$, $B$ | solar panels; battery energy (power $B/4$) | kW; kWh | decided; $0 \le S \le 4{,}000$, $0 \le B \le 8{,}000$ |
| $N$ | generator sets of 200 kW | whole sets, 0 to 4 | decided |
| $i, e, s, c, d$ | import, export, solar used, charge, discharge in an hour | kW | decided, $\ge 0$ |
| $q$ | battery stored at the end of the hour | kWh | decided, $\ge 0$ |
| $g$, $o$ | the sets' output; other loads served, each storm hour | kW | decided, $\ge 0$ |
| $a_S, a_B, a_N$ | each item's cost a year (capital × capital recovery factor at the owner's real rate, plus upkeep) | \$ per kW, kWh, set | given (`per_unit`) |
| $a_F$, $F$ | the tank's cost a year per gallon; diesel on site, 675 gallons | \$/gal; gal | given: $F$ is fixed, not sized |
| $w_m$, $\sigma$ | days in month $m$; sun per kW of panels | days; kW/kW | given |
| $r_h$, $x$ | tariff in hour $h$; export credit | \$/kWh: the data's ¢/kWh ÷ 100 | given |
| $\bar I$, $\bar E$ | the feeder's import and export limits | kW | given |
| $L^{\text{must}}$, $L^{\text{other}}$ | must-stay-on load (400 kW here); the other loads (800 kW) | kW | given |

$$\min\ a_S S + a_B B + a_N N + a_F F + \sum_m w_m \sum_h \big(r_h\, i - x\, e\big)$$

- every average hour: $s + d + i = c + e + 1{,}200$; $s \le \sigma S$; $c, d \le B/4$; $q \le B$; $i \le \bar I$; $e \le \bar E$
- every average day: $q_h = q_{h-1} + 0.9c_h - d_h$, with hour 0 following hour 23 (each day ends where it began)
- every storm hour, no grid: $s + d + g = c + o + L^{\text{must}}$; $s \le \sigma_t S$; $c, d \le B/4$; $q \le B$; $o \le L^{\text{other}}$; $g \le 200N$
- the storm: $q_t = q_{t-1} + 0.9c_t - d_t$ from $q_{-1} = B$ (a full battery); $\sum_t 0.07\,g_t \le F$

The sizing leaves out the sets' 30% minimum (section 5 puts it back). Small tie weights, as on the page: \$0.0001 per kWh through the battery in the year, and tiny fuel and battery terms in the storm (a few dollars at most).

In [ ]:
def must_kw(plan):
    return sum(l["kw"] for l in LOADS if plan["prio"][l["key"]] == "must")

def add_year(M, plan, size):
    credit = D["grid"]["exportCredit"] / 100
    for m in range(12):
        w = MD[m]
        for h in range(24):
            p = f"y{m}_{h}"; sun = SUN_MONTHS[m][h]
            M.var(p + "i", ub=plan["imp"], cost=w * rate(plan, h) / 100); M.var(p + "e", ub=plan["exp"], cost=-w * credit)
            M.var(p + "s", ub=math.inf if size else plan["pv"] * sun)
            M.var(p + "c", ub=math.inf if size else plan["bat"] / 4, cost=w * 1e-4); M.var(p + "d", ub=math.inf if size else plan["bat"] / 4, cost=w * 1e-4)
            M.var(p + "q", ub=math.inf if size else plan["bat"])
    for m in range(12):
        for h in range(24):
            p = f"y{m}_{h}"; prev = f"y{m}_{(h + 23) % 24}q"; sun = SUN_MONTHS[m][h]
            M.con({p + "s": 1, p + "d": 1, p + "i": 1, p + "c": -1, p + "e": -1}, TOTAL, TOTAL)
            M.con({p + "q": 1, prev: -1, p + "c": -T["bat"]["eff"], p + "d": 1}, 0, 0)
            if size:
                M.con({p + "s": 1, "S": -sun}, hi=0); M.con({p + "c": 1, "B": -0.25}, hi=0); M.con({p + "d": 1, "B": -0.25}, hi=0); M.con({p + "q": 1, "B": -1}, hi=0)

def size_model(plan, storms):
    M = Model(); L = D["limits"]; a = per_unit(plan); must = must_kw(plan); other = TOTAL - must
    M.var("S", ub=L["pv"][1], cost=a["pv"]); M.var("B", ub=L["bat"][1], cost=a["bat"]); M.var("N", ub=L["gen"][1], cost=a["gen"], integer=True)
    M.offset = a["tank"] * plan["fuel"]
    add_year(M, plan, size=True)
    for j, sun in enumerate(storms):
        for t in range(len(sun)):
            p = f"o{j}_{t}"
            M.var(p + "s"); M.var(p + "c", cost=1e-6); M.var(p + "d", cost=1e-6); M.var(p + "q"); M.var(p + "g", cost=1e-3 * T["gen"]["galPerKWh"]); M.var(p + "o", ub=other)
        fuel = {}
        for t in range(len(sun)):
            p = f"o{j}_{t}"
            M.con({p + "s": 1, p + "d": 1, p + "g": 1, p + "c": -1, p + "o": -1}, must, must)
            M.con({p + "q": 1, (f"o{j}_{t-1}q" if t else "B"): -1, p + "c": -T["bat"]["eff"], p + "d": 1}, 0, 0)
            M.con({p + "s": 1, "S": -sun[t]}, hi=0); M.con({p + "c": 1, "B": -0.25}, hi=0); M.con({p + "d": 1, "B": -0.25}, hi=0)
            M.con({p + "q": 1, "B": -1}, hi=0); M.con({p + "g": 1, "N": -T["gen"]["unitKW"]}, hi=0)
            fuel[p + "g"] = T["gen"]["galPerKWh"]
        M.con(fuel, hi=plan["fuel"])
    return M

def bill_of(plan, x):
    credit = D["grid"]["exportCredit"] / 100
    return sum(MD[m] * (rate(plan, h) / 100 * x[f"y{m}_{h}i"] - credit * x[f"y{m}_{h}e"]) for m in range(12) for h in range(24))

def size(plan, storms):
    M = size_model(plan, storms); r = checked(M.solve(), "the sizing")
    x = r["x"]; a = per_unit(plan); z = {"pv": x["S"], "bat": x["B"], "gen": round(x["N"])}
    z["total"] = a["pv"] * z["pv"] + a["bat"] * z["bat"] + a["gen"] * z["gen"] + a["tank"] * plan["fuel"] + bill_of(plan, x)
    return z

def case(exp, tar, storm):
    p = json.loads(json.dumps(BASE)); p["exp"] = exp; p["tar"] = tar
    return p, ([STORM] if storm else [])

print(f"{'case':42} {'solar kW':>9} {'battery kWh':>12} {'sets':>5} {'annual cost':>13}   page")
SIZED = {}
for c in REF["cases"]:
    plan, storms = case(c["exp"], c["tar"], c["storm"])
    z = size(plan, storms); SIZED[c["key"]] = z
    print(f"{c['key']:42} {z['pv']:9,.0f} {z['bat']:12,.0f} {z['gen']:5d} {z['total']:13,.0f}   ${c['total']:,.0f}")
    assert z["gen"] == c["gen"], "whole sets differ from the page"
    assert abs(z["total"] - c["total"]) < 5, "annual cost differs from the page by more than $5"
    assert abs(z["pv"] - c["pv"]) < 5 and abs(z["bat"] - c["bat"]) < 5, "sizes differ from the page"
storm_cost = SIZED["x0-tou-storm"]["total"] - SIZED["x0-tou-none"]["total"]
print(f"carrying the hospital and its water plant through the storm (time-of-use, exports capped): ${storm_cost:,.0f} a year, the difference of two solved plans")

## 4. Branch and bound on the sets, by hand

The relaxation lets $N$ be any number from 0 to 4. Its cost is a lower bound on every plan with whole sets. Branch on $N$: one child with $N \le \lfloor N^* \rfloor$, one with $N \ge \lceil N^* \rceil$; solve both relaxations; keep the cheaper whole-number plan, and prune any node whose bound cannot beat it.

In [ ]:
plan0, storms0 = case(2000, "flat", True)
M = size_model(plan0, storms0)
root = checked(M.solve(relax=True), "the root relaxation"); Nstar = root["x"]["N"]
print(f"root relaxation: {Nstar:.4f} sets, ${root['objective']:,.0f} a year (a lower bound)")
kids = []
for lo, hi in [(0, math.floor(Nstar)), (math.ceil(Nstar), D["limits"]["gen"][1])]:
    Mk = size_model(plan0, storms0); j = Mk.idx["N"]; Mk.lb[j], Mk.ub[j] = lo, hi
    r = checked(Mk.solve(relax=True), f"the node N in [{lo}, {hi}]"); kids.append((lo, hi, r))
    print(f"  N in [{lo}, {hi}]: {r['x']['N']:.4f} sets, ${r['objective']:,.0f}")
best = min((k for k in kids if abs(k[2]["x"]["N"] - round(k[2]["x"]["N"])) < 1e-6), key=lambda k: k[2]["objective"])
print(f"best whole-set plan: {round(best[2]['x']['N'])} sets, ${best[2]['objective']:,.0f}; both children are whole, so the gap is 0: proven after 1 branching and 3 LPs")
assert abs(Nstar - REF["tree"]["rootN"]) < 1e-3 and abs(root["objective"] - REF["tree"]["root"]) < 5
assert round(best[2]["x"]["N"]) == REF["cases"][0]["gen"] and REF["tree"]["branchings"] == 1 and REF["tree"]["lps"] == 3

## 5. The check: whole sets that run at 30% or more, hour by hour

Fix the sizes, rounded up to the bench's steps (25 kW, 50 kWh), as the page's step 6 does. Each storm hour: sets on $u_t \in \{0, \dots, N\}$, with $60\,u_t \le g_t \le 200\,u_t$; every load can be partly served.

The objective weighs unserved kWh, not money: 1,000 (must stay on), 10 (keep if possible), 1 (shed first). Within a priority, 0.5% more a step up the list and up to 2% more for earlier hours, so an earlier hour can outweigh the list; then 0.01 per gallon and 0.0001 per kWh through the battery.

In [ ]:
W = {"must": 1000, "keep": 10, "shed": 1}

def outage_model(plan, sun, init=None, tie=None):
    tie = dict({"order": 0.005, "time": 0.02, "fuel": 0.01, "battery": 1e-4}, **(tie or {}))
    init = init or {}; q0 = init.get("soc", plan["bat"]); f0 = init.get("fuel", plan["fuel"]); H = len(sun); U = T["gen"]["unitKW"]
    M = Model()
    for t in range(H):
        p = f"t{t}"
        M.var(p + "s", ub=plan["pv"] * sun[t]); M.var(p + "c", ub=plan["bat"] / 4, cost=tie["battery"]); M.var(p + "d", ub=plan["bat"] / 4, cost=tie["battery"])
        M.var(p + "q", ub=plan["bat"]); M.var(p + "g", ub=U * plan["gen"], cost=tie["fuel"] * T["gen"]["galPerKWh"]); M.var(p + "u", ub=plan["gen"], integer=True)
        for k, l in enumerate(LOADS):
            M.var(p + f"x{k}", ub=l["kw"], cost=W[plan["prio"][l["key"]]] * (1 + tie["order"] * (len(LOADS) - 1 - k)) * (1 + tie["time"] * (H - t) / H))
    fuel = {}
    for t in range(H):
        p = f"t{t}"
        bal = {p + "s": 1, p + "d": 1, p + "g": 1, p + "c": -1}; bal.update({p + f"x{k}": 1 for k in range(len(LOADS))})
        M.con(bal, TOTAL, TOTAL)
        if t: M.con({p + "q": 1, f"t{t-1}q": -1, p + "c": -T["bat"]["eff"], p + "d": 1}, 0, 0)
        else: M.con({p + "q": 1, p + "c": -T["bat"]["eff"], p + "d": 1}, q0, q0)
        M.con({p + "g": 1, p + "u": -U}, hi=0); M.con({p + "g": 1, p + "u": -U * T["gen"]["minLoad"]}, lo=0)
        fuel[p + "g"] = T["gen"]["galPerKWh"]
    M.con(fuel, hi=f0)
    return M

def read_outage(plan, sun, x):
    hours, tot = [], {"must": 0.0, "keep": 0.0, "shed": 0.0, "fuel": 0.0}
    for t in range(len(sun)):
        p = f"t{t}"; un = [max(0.0, x[p + f"x{k}"]) for k in range(len(LOADS))]
        for k, l in enumerate(LOADS): tot[plan["prio"][l["key"]]] += un[k]
        tot["fuel"] += T["gen"]["galPerKWh"] * x[p + "g"]
        hours.append({"u": round(x[p + "u"]), "g": x[p + "g"], "x": un, "q": x[p + "q"]})
    return hours, tot

def up(v, step, hi): return min(hi, math.ceil(v / step - 1e-6) * step)
z = SIZED["x0-tou-storm"]; L = D["limits"]
PLAN = json.loads(json.dumps(BASE)); PLAN.update({"exp": 0, "tar": "tou", "pv": up(z["pv"], L["pv"][2], L["pv"][1]), "bat": up(z["bat"], L["bat"][2], L["bat"][1]), "gen": z["gen"]})
assert {k: PLAN[k] for k in ("pv", "bat", "gen")} == REF["plan"], "the rounded plan differs from the page's"
r = checked(outage_model(PLAN, STORM).solve(gap=1e-6), "the storm check")
hours, tot = read_outage(PLAN, STORM, r["x"])
print(f"plan {PLAN['pv']:,} kW, {PLAN['bat']:,} kWh, {PLAN['gen']} set(s): must-stay-on unserved {tot['must']:,.0f} kWh, keep {tot['keep']:,.0f}, shed {tot['shed']:,.0f}, diesel {tot['fuel']:,.1f} gal ({tot['fuel'] * D['fuel']['co2'] / 1000:.1f} t CO2)")
assert abs(tot["must"] - REF["check"]["must"]) < 1 and abs(tot["keep"] - REF["check"]["keep"]) < 1 and abs(tot["fuel"] - REF["check"]["fuel"]) < 0.5
assert all(h["u"] == 0 or 60 * h["u"] - 1e-6 <= h["g"] <= 200 * h["u"] + 1e-6 for h in hours), "a set ran below its 30% minimum"

## 6. Ten drawn winter outages

The page draws them with a small seeded generator (mulberry32, seed 7): each starts at a whole hour from 00:00 on 1 January to 23:00 on 25 February 2025 and lasts 6 to 72 hours. This cell draws the same ten and runs the plan through each. Passing them is ten tests passed, not a failure rate.

In [ ]:
def mulberry32(seed):
    a = seed & 0xFFFFFFFF
    def r():
        nonlocal a
        a = (a + 0x6D2B79F5) & 0xFFFFFFFF
        t = a
        t = ((t ^ (t >> 15)) * (t | 1)) & 0xFFFFFFFF
        t = (t ^ ((t + (((t ^ (t >> 7)) * (t | 61)) & 0xFFFFFFFF)) & 0xFFFFFFFF)) & 0xFFFFFFFF
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296
    return r

S = D["seeded"]; rnd = mulberry32(S["seed"]); first, last = S["firstDay"] * 24, S["lastStartDay"] * 24 + 23
DRAWN = []
for i in range(S["n"]):
    start = first + math.floor(rnd() * (last - first + 1)); hours_n = S["minHours"] + math.floor(rnd() * (S["maxHours"] - S["minHours"] + 1))
    DRAWN.append((start, hours_n))
assert [d[0] for d in DRAWN] == REF["seeded"]["starts"] and [d[1] for d in DRAWN] == REF["seeded"]["hours"], "the draws differ from the page's"
served = 0
for i, (start, n) in enumerate(DRAWN):
    sun = WINTER[start: start + n]
    _, tt = read_outage(PLAN, sun, checked(outage_model(PLAN, sun).solve(gap=1e-6), f"drawn outage #{i + 1}")["x"])
    served += tt["must"] < 0.5
    print(f"#{i + 1:2d}  day {start // 24 + 1:2d} {start % 24:02d}:00  {n:2d} h   must-stay-on unserved {tt['must']:8,.0f} kWh  (page {REF['seeded']['must'][i]:,.0f})")
    assert abs(tt["must"] - REF["seeded"]["must"][i]) < 1
print(f"fully served in {served} of {len(DRAWN)} drawn outages")

## 7. The storm on forecasts: a rolling horizon

Every 6 hours the controller:

1. re-plans the rest of the storm on a forecast of the sun: the same outage model, from the battery's charge and the diesel left;
2. keeps that plan's sets on or off for the next 6 hours;
3. balances each real hour against the real sun, as follows.

In each real hour:

1. All the sun goes in first, then the planned sets' output.
2. Short: the battery, then the sets up to full, then loads are shed: shed-first loads first, must-stay-on loads last.
3. Spare: shed must-stay-on loads come back first, then the sets go down to their minimum, the battery charges, the other loads come back, and only then is solar curtailed.

The losses below hold for this controller; another rule would lose differently.

Forecasts: perfect; yesterday again (each coming day's sun is the last full day's); too sunny (yesterday's × 1.5, at most the panels' size); too dull (× 0.5).

In [ ]:
def forecast(t0, key):
    if key == "perfect": return STORM[t0:]
    f = next(x for x in D["forecasts"] if x["key"] == key); start = 19 * 24; last_day = (start + t0) // 24 - 1
    out = [WINTER[last_day * 24 + (start + t) % 24] for t in range(t0, len(STORM))]
    return [min(1.0, v * f["factor"]) for v in out] if f.get("factor") else out

def real_hour(plan, ph, sun_real, st):
    U, eta, gpk = T["gen"]["unitKW"], T["bat"]["eff"], T["gen"]["galPerKWh"]
    u = max(0, min(plan["gen"], round(ph["u"])))
    while u > 0 and st["fuel"] < U * T["gen"]["minLoad"] * u * gpk - 1e-9: u -= 1
    gmin, gmax = U * T["gen"]["minLoad"] * u, min(U * u, st["fuel"] / gpk)
    order = [k for c in ("shed", "keep", "must") for k in reversed(range(len(LOADS))) if plan["prio"][LOADS[k]["key"]] == c]
    serve = [max(0.0, min(l["kw"], l["kw"] - ph["x"][k])) for k, l in enumerate(LOADS)]
    s = plan["pv"] * sun_real; g = max(gmin, min(gmax, ph["g"])); net = sum(serve) - s - g; c = d = 0.0
    dmax, cmax = min(plan["bat"] / 4, st["soc"]), min(plan["bat"] / 4, (plan["bat"] - st["soc"]) / eta)
    if net > 1e-9:
        d = min(net, dmax); net -= d
        more = min(net, gmax - g); g += more; net -= more
        for k in order:
            if net <= 1e-9: break
            cut = min(serve[k], net); serve[k] -= cut; net -= cut
    elif net < -1e-9:
        spare = -net
        for k in reversed(order):   # shed must-stay-on loads come back first
            if spare <= 1e-9: break
            if plan["prio"][LOADS[k]["key"]] != "must": continue
            add = min(LOADS[k]["kw"] - serve[k], spare); serve[k] += add; spare -= add
        down = min(spare, g - gmin); g -= down; spare -= down
        c = min(spare, cmax); spare -= c
        for k in reversed(order):
            if spare <= 1e-9: break
            add = min(LOADS[k]["kw"] - serve[k], spare); serve[k] += add; spare -= add
        s -= min(spare, s)
    st2 = {"soc": min(plan["bat"], max(0.0, st["soc"] + eta * c - d)), "fuel": max(0.0, st["fuel"] - gpk * g)}
    # the hour's physics: supply meets the loads served, the battery and the tank stay in bounds, a set that is on runs at 30% to 100%
    served = sum(serve)
    assert abs(s + d + g - c - served) < 1e-6, "a real hour does not balance"
    assert -1e-9 <= st2["soc"] <= plan["bat"] + 1e-9 and st2["fuel"] >= -1e-9 and s <= plan["pv"] * sun_real + 1e-9
    assert u == 0 or U * T["gen"]["minLoad"] * u - 1e-6 <= g <= U * u + 1e-6, "a set ran outside 30% to 100%"
    return [max(0.0, l["kw"] - serve[k]) for k, l in enumerate(LOADS)], g, st2

def roll(plan, key, every=6):
    st, tot = {"soc": plan["bat"], "fuel": plan["fuel"]}, {"must": 0.0, "keep": 0.0, "shed": 0.0, "fuel": 0.0}
    for t0 in range(0, len(STORM), every):
        fc = forecast(t0, key); r = checked(outage_model(plan, fc, init=st).solve(gap=1e-6), f"the re-plan at storm hour {t0} ({key})", provisional=True)
        ph_hours, _ = read_outage(plan, fc, r["x"])
        for t in range(t0, min(len(STORM), t0 + every)):
            x, g, st = real_hour(plan, ph_hours[t - t0], STORM[t], st)
            for k, l in enumerate(LOADS): tot[plan["prio"][l["key"]]] += x[k]
            tot["fuel"] += T["gen"]["galPerKWh"] * g
    return tot

page = {r["key"]: r for r in REF["roll"]}
for f in D["forecasts"]:
    tt = roll(PLAN, f["key"])
    print(f"{f['label']:16} must-stay-on unserved {tt['must']:8,.0f} kWh (page {page[f['key']]['must']:8,.0f}) · keep {tt['keep']:7,.0f} · diesel {tt['fuel']:6,.0f} gal")

## 8. Your plan (the Modeler version of capstone C1)

Set the sizes and priorities of your C1 plan in the cell below (the one that starts `MY = `), run it through the storm and the ten drawn outages, and hand in the table with your note. The cells above reproduce the lesson's reference cases: leave them as they are. The dorms must stay on in C1's brief.

The table counts the brief's three critical loads (the hospital, its water and heating plant, the dorms) one by one, whatever priorities you set, and says whether the plan meets the brief: all three served through the storm.

In [ ]:
MY = json.loads(json.dumps(BASE))
MY.update({"pv": 600, "bat": 2000, "gen": 2, "fuel": 675, "tar": "tou", "exp": 500})   # your sizes, diesel, tariff and export limit
MY["prio"].update({"dorms": "must"})                                                 # C1: power to the hospital, its water and heating plant and the dorms
CRIT = ["hospital", "water", "dorms"]   # C1's brief, counted by load whatever MY["prio"] says (not by priority label)
def critical(hours):                     # unserved kWh of each critical load over an outage
    return {l["key"]: sum(h["x"][k] for h in hours) for k, l in enumerate(LOADS) if l["key"] in CRIT}
NAME = {l["key"]: l["short"] for l in LOADS}
hs, tt = read_outage(MY, STORM, checked(outage_model(MY, STORM).solve(gap=1e-6), "your plan in the storm")["x"])
cs = critical(hs)
print(f"the storm ({len(STORM)} h), unserved: " + ", ".join(f"{NAME[k]} {cs[k]:,.0f} kWh" for k in CRIT) + f"; critical in all {sum(cs.values()):,.0f} kWh")
print(f"           must-stay-on (your priorities) {tt['must']:,.0f} kWh, diesel {tt['fuel']:,.0f} gal")
print("C1's brief (the hospital, its water and heating plant and the dorms served through the storm):", "met" if sum(cs.values()) < 0.5 else "NOT met")
print(" #   hours   critical unserved   must-stay-on unserved")
for i, (start, n) in enumerate(DRAWN):
    h2, t2 = read_outage(MY, WINTER[start: start + n], checked(outage_model(MY, WINTER[start: start + n]).solve(gap=1e-6), "your plan, a drawn outage")["x"])
    print(f"#{i + 1:2d}  {n:3d} h   {sum(critical(h2).values()):10,.0f} kWh   {t2['must']:14,.0f} kWh")

## Practice (about 90 minutes)

1. **The cap on exports.** Size the time-of-use case with exports capped at 250 kW and at 1,000 kW. At what cap does solar stop growing? Explain with one hour of a July day.
2. **The minimum, back in the sizing.** Add $u_t$ and $60u_t \le g_t \le 200u_t$ to the sizing model of section 3 for the storm. Does the plan change? How long does it take to solve?
3. **Part-load fuel.** Replace $0.07\,g_t$ by $1.5\,u_t + 0.063\,g_t$ gallons (a no-load term). Re-run the check: does the plan still carry the must-stay-on loads? What happens to the solve time?
4. **A margin for forecasts.** Size the plan for the storm with 10% more must-stay-on load, then run the rolling horizon on "yesterday again". What does the margin cost a year, and what does it buy?
5. **Owners.** Size the storm case for the utility (6%), the co-op (4%) and the community (5%). Which sizes change, and why does the cost change even when the sizes do not?